# Estimasi Klorofil-a Segara Anakan, Cilacap (komposit musiman 2023-2025)

UAS Sistem Informasi Geografi, Opsi B (Ecological Layer). Notebook ini mengestimasi klorofil-a dari citra Sentinel-2 di Google Earth Engine, lalu membandingkan musim hujan dan kemarau selama tiga tahun. Hasilnya berupa tabel statistik, grafik bulanan, dan peta musiman.

Jalankan sel dari atas ke bawah dan isi `PROJECT_ID` di sel parameter. Setelah peta satelit muncul, cek kotak laguna sebelum lanjut.

## Setup

In [ ]:
!pip -q install geemap rasterio
import ee, geemap
ee.Authenticate()


## Parameter

In [ ]:
PROJECT_ID = 'NAMA-PROJECT-GEE-ANDA'   # <-- GANTI
ee.Initialize(project=PROJECT_ID)

# Area studi (lebih lebar ke barat) dan zona laguna: [barat, selatan, timur, utara]
ROI_BOX    = [108.60, -7.82, 109.02, -7.60]
LAGOON_BOX = [108.60, -7.80, 108.95, -7.62]   # <-- sesuaikan setelah cek sel 4
roi    = ee.Geometry.Rectangle(ROI_BOX)
lagoon = ee.Geometry.Rectangle(LAGOON_BOX)

YEARS = [2023, 2024, 2025]
WET = [12, 1, 2, 3]      # musim hujan (perkiraan)
DRY = [6, 7, 8, 9]       # musim kemarau (perkiraan)
CLOUD_MAX = 60           # batas % awan per scene
MIN_OBS = 3              # minimal observasi valid per piksel
CHL_MAX = 60             # nilai di atas ini dianggap tidak masuk akal
VIS = {'min': 0, 'max': 40,
       'palette': ['#00429d', '#4ac16d', '#fde725', '#ff6f3c', '#b2182b']}
SCALE = 30               # resolusi analisis dan ekspor (m)


## Fungsi pengolahan

In [ ]:
base = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(roi)
        .filterDate(f'{YEARS[0]}-01-01', f'{YEARS[-1]}-12-31')
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_MAX))
        .map(lambda i: i.set('month', i.date().get('month'))))

def prep(img):
    scl = img.select('SCL')
    clear = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                .And(scl.neq(10)).And(scl.neq(11)))
    sr = img.select(['B3', 'B4', 'B5', 'B8']).divide(10000)
    ndwi = sr.normalizedDifference(['B3', 'B8'])
    ndci = sr.normalizedDifference(['B5', 'B4'])
    chl = ndci.expression('14.039 + 86.115*x + 194.325*x*x', {'x': ndci}).rename('chl_a')
    ok = clear.And(ndwi.gt(0.1)).And(chl.gte(0)).And(chl.lte(CHL_MAX))
    return chl.updateMask(ok).copyProperties(img, ['month'])

chl_col = base.map(prep)

def season_img(months):
    col = chl_col.filter(ee.Filter.inList('month', months))
    return col.median().updateMask(col.count().gte(MIN_OBS)).clip(roi)

def n_scene(months):
    return base.filter(ee.Filter.inList('month', months)).size().getInfo()

wet_img, dry_img = season_img(WET), season_img(DRY)
N_WET, N_DRY = n_scene(WET), n_scene(DRY)
print(f'Scene musim hujan   : {N_WET}')
print(f'Scene musim kemarau : {N_DRY}')


## Cek area studi dan zona laguna

Kotak kuning adalah zona laguna. Kalau masih ikut laut terbuka atau belum menutup seluruh laguna, ubah `LAGOON_BOX` di sel parameter lalu jalankan ulang dari sel itu.

In [ ]:
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.centerObject(roi, 10)
Map.addLayer(ee.Image().paint(roi, 0, 2), {'palette': 'white'}, 'Area studi')
Map.addLayer(ee.Image().paint(lagoon, 0, 2), {'palette': 'yellow'}, 'Zona laguna')
Map.addLayer(wet_img, VIS, 'Chl-a musim hujan')
Map.addLayer(dry_img, VIS, 'Chl-a musim kemarau')
Map.add_colorbar(VIS, label='Chl-a estimasi (mg/m³)')
Map.addLayerControl()
Map


## Statistik per musim

In [ ]:
import pandas as pd

reducer = (ee.Reducer.mean().combine(ee.Reducer.median(), sharedInputs=True)
           .combine(ee.Reducer.percentile([10, 90]), sharedInputs=True)
           .combine(ee.Reducer.count(), sharedInputs=True))

def zstats(img, geom):
    r = img.reduceRegion(reducer, geom, SCALE, maxPixels=1e9).getInfo()
    return {k.replace('chl_a_', ''): (round(v, 2) if v is not None else None) for k, v in r.items()}

rows = []
for zona, geom in (('Seluruh area', roi), ('Zona laguna', lagoon)):
    for musim, img, n in (('Hujan', wet_img, N_WET), ('Kemarau', dry_img, N_DRY)):
        rows.append({'zona': zona, 'musim': musim, 'scene': n, **zstats(img, geom)})
df_stat = pd.DataFrame(rows)
df_stat.to_csv('statistik_musiman.csv', index=False)
df_stat


## Pola bulanan di zona laguna

In [ ]:
import matplotlib.pyplot as plt

rows = []
for m in range(1, 13):
    col = chl_col.filter(ee.Filter.inList('month', [m]))
    n = base.filter(ee.Filter.inList('month', [m])).size().getInfo()
    v = None
    if n > 0:
        v = (col.median().clip(roi)
             .reduceRegion(ee.Reducer.mean(), lagoon, SCALE, maxPixels=1e9)
             .get('chl_a').getInfo())
    rows.append({'bulan': m, 'scene': n, 'chl_a_laguna': v})
df_bln = pd.DataFrame(rows)
df_bln.to_csv('chla_bulanan_laguna_2023_2025.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
for m in WET: ax.axvspan(m - .5, m + .5, color='#4a90d9', alpha=.12)
for m in DRY: ax.axvspan(m - .5, m + .5, color='#e0a030', alpha=.15)
ax.plot(df_bln['bulan'], df_bln['chl_a_laguna'], marker='o', color='#14797F')
for _, r in df_bln.iterrows():
    if pd.notna(r['chl_a_laguna']):
        ax.annotate(f"n={int(r['scene'])}", (r['bulan'], r['chl_a_laguna']),
                    textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax.set_xticks(range(1, 13)); ax.set_xlabel('Bulan')
ax.set_ylabel('Chl-a estimasi (mg/m³)'); ax.grid(alpha=.3)
ax.set_title(f'Rata-rata Chl-a bulanan, zona laguna Segara Anakan ({YEARS[0]}-{YEARS[-1]})')
ax.text(.01, .02, 'Biru = musim hujan, oranye = musim kemarau; n = jumlah scene', transform=ax.transAxes, fontsize=8)
plt.tight_layout(); plt.savefig('grafik_chla_bulanan_v2.png', dpi=300); plt.show()
df_bln


## Peta musiman

In [ ]:
import numpy as np, rasterio, math
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

cmap = LinearSegmentedColormap.from_list('chl', VIS['palette'])
norm = Normalize(VIS['min'], VIS['max'])
NODATA = -9999

def download(img, fn):
    geemap.ee_export_image(img.toFloat().unmask(NODATA), filename=fn, scale=SCALE,
                           region=roi, crs='EPSG:4326', file_per_band=False)
    return fn

def draw(ax, fn, title):
    with rasterio.open(fn) as src:
        a = src.read(1).astype(float); b = src.bounds
    a[a == NODATA] = np.nan
    ax.imshow(np.ma.masked_invalid(a), cmap=cmap, norm=norm,
              extent=[b.left, b.right, b.bottom, b.top])
    ax.set_facecolor('#e9e9e9')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Bujur (°BT)'); ax.set_ylabel('Lintang (°LS)')
    ax.grid(alpha=.3, ls='--')
    w, s, e, n = LAGOON_BOX                       # batas zona laguna
    ax.plot([w, e, e, w, w], [s, s, n, n, s], 'k--', lw=1.2)
    ax.text(w + .005, n - .012, 'Zona laguna', fontsize=8, fontweight='bold')
    ax.annotate('N', xy=(.93, .93), xytext=(.93, .80), xycoords='axes fraction',
                ha='center', fontsize=12, fontweight='bold',
                arrowprops=dict(arrowstyle='-|>', lw=2, color='k'))
    lat = (b.top + b.bottom) / 2
    deg = 5 / (111.32 * math.cos(math.radians(lat)))
    x0 = b.left + (b.right - b.left) * .05
    y0 = b.bottom + (b.top - b.bottom) * .06
    ax.plot([x0, x0 + deg], [y0, y0], 'k-', lw=4)
    ax.text(x0 + deg / 2, y0 + (b.top - b.bottom) * .02, '5 km', ha='center', fontsize=9)

f1 = download(wet_img, 'chla_musim_hujan.tif')
f2 = download(dry_img, 'chla_musim_kemarau.tif')
fig, axs = plt.subplots(1, 2, figsize=(15, 6))
draw(axs[0], f1, f'Musim hujan (bulan {WET}; {N_WET} scene)')
draw(axs[1], f2, f'Musim kemarau (bulan {DRY}; {N_DRY} scene)')
fig.suptitle(f'Estimasi Klorofil-a Segara Anakan, Cilacap, komposit median {YEARS[0]}-{YEARS[-1]}',
             fontsize=14, fontweight='bold')
cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=axs, shrink=.75, pad=.02)
cb.set_label('Klorofil-a estimasi (mg/m³)')
fig.text(.01, .01, f'Sumber: Sentinel-2 SR Harmonized (ESA/Copernicus) via GEE; Chl-a = NDCI (Mishra & Mishra, 2012); '
         f'piksel dengan < {MIN_OBS} observasi valid dibuang. WGS 84 (EPSG:4326).', fontsize=8)
plt.savefig('peta_chla_musiman_v2.png', dpi=300, bbox_inches='tight'); plt.show()


## Catatan

Persamaan Chl-a belum divalidasi dengan data lapangan, jadi hasilnya dibaca sebagai estimasi relatif. Pembagian musim (Des-Mar dan Jun-Sep) adalah perkiraan, dan batas zona laguna berupa kotak kasar.